<a href="https://colab.research.google.com/github/rimaoucherif/FairnessIA-Project-Bias-Mitigation/blob/main/UPP26/train_classifieur_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Only for Google Colab
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
# Only for Google Colab
import sys

# In your google drive, create a folder for the Fairness project
# (called here 'coursFairness')
# Copy in this folder the python file train_classifieur.py
# Copy in this folder your zipped data folder (called here 'YOUR_NAME.zip')

# Path to the folder containing the python file
drive_module_path = "/content/drive/MyDrive/coursFairness"
sys.path.append(drive_module_path)

# Verify the path was added
print(sys.path[-1])

/content/drive/MyDrive/coursFairness


In [4]:
# Only for Google Colab
!cp /content/drive/MyDrive/coursFairness/YOUR_NAME.zip /content/
!mkdir /content/DATA/
!unzip YOUR_NAME.zip -d /content/DATA/

cp: cannot stat '/content/drive/MyDrive/coursFairness/YOUR_NAME.zip': No such file or directory
unzip:  cannot find or open YOUR_NAME.zip, YOUR_NAME.zip.zip or YOUR_NAME.zip.ZIP.


In [5]:
# Only for Google Colab
!pip install pandas\
  numpy\
  torchvision\
  torch\
  torchmetrics\
  tensorboard\
  pillow\
  lightning\
  matplotlib\
  scikit_learn\
  ipykernel \
  fairlearn \
  plotly \
  nbformat \
  aif360["inFairness"] \
  aif360['AdversarialDebiasing'] \
  causal-learn \
  BlackBoxAuditing \
  cvxpy \
  dice-ml \
  lime \
  shapkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 20.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 29.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
from google.colab import files
files.upload()

Saving train_classifieur.py to train_classifieur.py


{'train_classifieur.py': b'from typing import Any\nimport argparse\nfrom torchvision.models import resnet18, ResNet18_Weights\nimport torch\nfrom torch.utils.data import DataLoader, WeightedRandomSampler\nfrom torch.nn.functional import cross_entropy\nfrom torchmetrics.classification import BinaryConfusionMatrix, ConfusionMatrix\nfrom torchvision.transforms.v2 import (\n    Compose,\n    Resize,\n    CenterCrop,\n    ToImage,\n    ToDtype,\n    Normalize,\n    RandomCrop,\n    RandomHorizontalFlip,\n)\nfrom torchvision.datasets import ImageFolder\nfrom pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping\nfrom pytorch_lightning.loggers import TensorBoardLogger\nfrom pytorch_lightning import LightningModule, Trainer\nfrom sklearn.metrics import balanced_accuracy_score, accuracy_score\n\nimport io\nimport shutil\nimport os\nimport pandas as pd\nfrom time import time\nimport matplotlib.pyplot as plt\nfrom PIL import Image\nimport numpy as np\nfrom tqdm import tqdm\n\n\n\ndef 

In [8]:
from train_classifieur import train_classifier, pred_classifier
from datetime import datetime

In [9]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [10]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

Tesla T4
Memory Usage:
Allocated: 0.0 GB
Cached:    0.0 GB


In [ ]:
from google.colab import files
files.upload()

In [ ]:
%%time
# This cell show how to train a model
#
# You are going to train several time the models, with different weights
#
# The experiment folder name (logdir) suggested contains the timestamp
# you can change this to add some description in the name
# or add a file in the logdir folder to describe your intention and parameters
# Warning: For Colab users, the logdir needs to be on your drive (as in this example)
# It will allow you to keep your trained models, even if you get disconnected
#
# By default, to help you reproduce you experiments the csv file
# used for the training is copied in the logdir folder.

# This can take around 20-30min to run on PUIO computer
# And around 10 min on Colab with a GPU

dt = datetime.now()
timeStamp = dt.strftime('%Y_%m_%d_%H_%M_%S')
logdir = f"{drive_module_path}/expe_log/{timeStamp}"
ckpt_path, ckpt_score = train_classifier(
    logdir=logdir,
    datadir=f"./DATA/",
    csv=f"./DATA/metadata.csv",
    weights_col="WEIGHTS")

In [ ]:
%load_ext tensorboard
%tensorboard --logdir=$logdir

In [ ]:
%%time
# This cell show how to perform predictions with a train model
# the ckpt_path outputed in the previous cell refer to the best model obtained
# during the training, you can replace this by any .ckpt file obtained
pred_classifier(
    datadir=f"./DATA/",
    csv_in=f"./DATA/metadata.csv",
    csv_out=f"{logdir}/preds.csv",
    ckpt_path = ckpt_path
)